# Rental Car Allocation 

###  Bus 36109 "Advanced Decision Modeling with Python", Don Eisenstein
Don Eisenstein &copy; Copyright 2022, University of Chicago 

---

Consider a rental car location.  We wish to estimate the optimal number of rental cars to locate at the facility.  

- Each car costs \$10/day at the location whether it is rented or not we call inventory cost (cost of capital, upkeep, insurance, etc.)
- Each car earns revenue at a rate of \$40/day when rented (fraction of days are computed accordingly).
- If a customer demands a car, but none are available, the customer is lost to a competitor at a one-time cost of \$80.
- Assume all cars are of the same type
- Consider the facility to be open continuously, 24 hrs/day.
- Time between customer arrivals to demand a single car follow an exponential distribution with mean of 10 minutes. 
- The length of time cars are rented is considered to follow an exponential distribution with a mean of 3 days. 

Estimate the optimal number of cars to locate at the facility.

In [1]:
# Define some parameters 
MEAN_TIME_BETWEEN_DEMAND_ARRIVALS = 10  # Minutes
MEAN_RENTAL_DURATION = 3 * 24 * 60   # Minutes
DAILY_INVENTORY_COST = 10
DAILY_RENTAL_REV = 40
LOST_SALE_COST = 80

In [2]:
import numpy as np
import simpy
from simpy_helpers import Entity, Resource, Source, Stats

In [3]:
NUM_SIMULATED_DAYS = 30

In [4]:
NUM_CARS = 400

In [12]:
class GenerateDemand(Source):

    def interarrival_time(self):
        return np.random.exponential(MEAN_TIME_BETWEEN_DEMAND_ARRIVALS) # minutes

    def build_entity(self):
        attributes = {}
        return Customer(env, attributes)

In [7]:
class CarLot(Resource):

    def service_time(self, entity):
        return np.random.exponential(MEAN_RENTAL_DURATION)

In [25]:
class Customer(Entity):

    def process(self):
        # print(f'> Now processing {self.name}')
        # print(f'  Rental queue length: {len(carlot.queue)}')
        # print(f'  Now, {carlot.count} out of {carlot.capacity} cars are being rented')
        
        if carlot.count < carlot.capacity:
            self.attributes['outcome'] = 'rental'
            yield self.wait_for_resource(carlot)
            yield self.process_at_resource(carlot)
            self.release_resource(carlot)
        else:
            self.attributes['outcome'] = 'renege'
            # print(f'!!! {self.name} reneges') 

In [26]:
NUM_CARS = 400

np.random.seed(2429)
env = simpy.Environment()

carlot = CarLot(env, capacity=NUM_CARS)
source = GenerateDemand(env)

env.process(source.start(debug=False))
env.run(until=NUM_SIMULATED_DAYS * 24 * 60)

In [35]:
inventory_cost = NUM_SIMULATED_DAYS * DAILY_INVENTORY_COST * NUM_CARS
renege_cost = LOST_SALE_COST * len(Stats.get_processing_times(attributes={'outcome':'renege'}))
rental_revenue = DAILY_RENTAL_REV * np.sum(Stats.get_processing_times(carlot))/(24 * 60)

economic_profit = rental_revenue - renege_cost - inventory_cost
print(economic_profit)

246944.48445533466


In [45]:
np.random.seed(89)

for num_cars in range(50,1000,50):
    print('--------------------------------------------------')
    print(f'Number of cars: {num_cars}')
    
    env = simpy.Environment()

    carlot = CarLot(env, capacity=num_cars)
    source = GenerateDemand(env)
    
    env.process(source.start(debug=False))

    inventory_cost = NUM_SIMULATED_DAYS * DAILY_INVENTORY_COST * num_cars
    renege_cost = LOST_SALE_COST * len(Stats.get_processing_times(attributes={'outcome':'renege'}))
    rental_revenue = DAILY_RENTAL_REV * np.sum(Stats.get_processing_times(carlot))/(24 * 60)
    
    economic_profit = rental_revenue - renege_cost - inventory_cost
    print(f'Profit: {economic_profit}')

--------------------------------------------------
Number of cars: 50
Profit: 384571.341242829
--------------------------------------------------
Number of cars: 100
Profit: 369571.341242829
--------------------------------------------------
Number of cars: 150
Profit: 354571.341242829
--------------------------------------------------
Number of cars: 200
Profit: 339571.341242829
--------------------------------------------------
Number of cars: 250
Profit: 324571.341242829
--------------------------------------------------
Number of cars: 300
Profit: 309571.341242829
--------------------------------------------------
Number of cars: 350
Profit: 294571.341242829
--------------------------------------------------
Number of cars: 400
Profit: 279571.341242829
--------------------------------------------------
Number of cars: 450
Profit: 264571.341242829
--------------------------------------------------
Number of cars: 500
Profit: 249571.341242829
-----------------------------------------